# EV Charging & Grid Optimization - Data Cleaning and Preprocessing
**Author:** Min Thant (Data Loading & Cleaning)

This notebook focuses on the cleaning and preprocessing of the EV charging dataset. We implement robust imputation techniques (SimpleImputer and KNNImputer) to handle any potential missing values and prepare the data for downstream analysis and modeling.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder

In [3]:

# 1. Load the dataset
file_path = 'D:\\advancedML\\Datasets\\charging_ev_and_grid_optimization_dataset.csv'
df = pd.read_csv(file_path)

print(f"Initial Dataset Shape: {df.shape}")
df.head()

Initial Dataset Shape: (8354, 27)


,timestamp,station_id,location_type,vehicle_id,vehicle_type,arrival_time,charging_start_time,charging_end_time,waiting_time,battery_capacity_kWh,...,electricity_price,renewable_energy_ratio,traffic_density,weather_condition,day_of_week,time_slot,charging_demand,assigned_charger_id,charging_priority,optimization_reward
0,1/1/2025 0:00,ST004,Urban,EV10000,Two-Wheeler,1/1/2025 0:00,1/1/2025 0:12,1/1/2025 4:33,12,60,...,13.66,0.280335,Low,Cloudy,Wednesday,Off-Peak,17.242398,CH4,Low,-8.622299
1,1/1/2025 0:15,ST005,Urban,EV10001,Two-Wheeler,1/1/2025 0:15,1/1/2025 0:23,1/1/2025 1:12,8,100,...,5.47,0.392127,Low,Rainy,Wednesday,Off-Peak,18.324933,CH9,Low,-1.935644
2,1/1/2025 0:30,ST019,Highway,EV10002,Car,1/1/2025 0:30,1/1/2025 0:41,1/1/2025 1:35,11,75,...,9.50,0.103979,Low,Clear,Wednesday,Off-Peak,36.028168,CH2,Low,-18.201846
3,1/1/2025 0:45,ST008,Urban,EV10003,Two-Wheeler,1/1/2025 0:45,1/1/2025 0:54,1/1/2025 3:29,9,40,...,6.22,0.248553,Low,Clear,Wednesday,Off-Peak,17.146935,CH9,Medium,-7.404018
4,1/1/2025 1:00,ST008,Highway,EV10004,Two-Wheeler,1/1/2025 1:00,1/1/2025 1:08,1/1/2025 6:14,8,75,...,13.42,0.234926,Low,Cloudy,Wednesday,Off-Peak,14.577768,CH1,Low,-6.577466


### 2. Basic Cleaning and Sorting
We convert timestamps and sort the data chronologically.

In [4]:
date_cols = ["timestamp", "arrival_time", "charging_start_time", "charging_end_time"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df = df.sort_values("timestamp").reset_index(drop=True)
print("Datetime conversion and sorting complete.")

Datetime conversion and sorting complete.


### 3. Feature Selection
Removing unique identifiers that don't contribute to the time-series patterns.

In [5]:
cols_to_drop = ['vehicle_id', 'assigned_charger_id']
df_cleaned = df.drop(columns=cols_to_drop)
print(f"Dropped columns: {cols_to_drop}")

Dropped columns: ['vehicle_id', 'assigned_charger_id']


### 4. Data Imputation
Although the current dataset is clean (0 missing values), we implement a pipeline using `SimpleImputer` and `KNNImputer` to handle future data drift or missing entries in real-time streams.

In [6]:
# Identify numerical and categorical columns
num_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_cleaned.select_dtypes(exclude=[np.number, 'datetime64[ns]']).columns.tolist()

print(f"Numerical Columns: {num_cols}")
print(f"Categorical Columns: {cat_cols}")

# --- Step A: Simple Imputation ---
# Categorical: Fill with most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df_cleaned[cat_cols] = cat_imputer.fit_transform(df_cleaned[cat_cols])

# Numerical: Fill with median (robust to outliers)
num_imputer = SimpleImputer(strategy='median')
df_cleaned[num_cols] = num_imputer.fit_transform(df_cleaned[num_cols])

print("Simple Imputation complete.")

Numerical Columns: ['waiting_time', 'battery_capacity_kWh', 'initial_soc', 'final_soc', 'energy_consumed_kWh', 'charging_power_kW', 'charging_duration', 'queue_length', 'station_load', 'electricity_price', 'renewable_energy_ratio', 'charging_demand', 'optimization_reward']
Categorical Columns: ['station_id', 'location_type', 'vehicle_type', 'traffic_density', 'weather_condition', 'day_of_week', 'time_slot', 'charging_priority']
Simple Imputation complete.


In [7]:
# --- Step B: KNN Imputation (Advanced) ---
# KNN Imputer works best on numerical data. 
# It calculates the mean of the K-nearest neighbors to fill missing values.
knn_imputer = KNNImputer(n_neighbors=5)
df_cleaned[num_cols] = knn_imputer.fit_transform(df_cleaned[num_cols])

print("KNN Imputation complete (applied to numerical columns).")

KNN Imputation complete (applied to numerical columns).


### 5. Final Output
Saving the preprocessed dataset for Simon's EDA and Liam's Modelling.

In [8]:
output_file = 'preprocessed_ev_data.csv'
df_cleaned.to_csv(output_file, index=False)
print(f"Successfully saved preprocessed data to {output_file}")

Successfully saved preprocessed data to preprocessed_ev_data.csv
